# Digits memory store

Download once, then run fully offline. Storage:

- `digits.db` — SQLite, persistent on disk.
- in-memory dict (`bytes(features) -> label`) for O(1) exact lookup.
- in-memory KDTree for O(log n) nearest neighbor when no exact match.

In [ ]:
import os, sqlite3, time
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neighbors import KDTree

HERE = os.getcwd()
DB_PATH = os.path.join(HERE, 'digits.db')
CSV_PATH = os.path.join(HERE, 'digits.csv')
N_FEATURES = 64

## 1. Download once and persist

After the first run, the dataset lives in `digits.db` and `digits.csv`. The cell is a no-op on later runs.

In [ ]:
def build_store(force=False):
    if os.path.exists(DB_PATH) and os.path.exists(CSV_PATH) and not force:
        print(f'Already cached at {DB_PATH}')
        return
    print('Downloading digits dataset (one-time)...')
    digits = load_digits()
    X = digits.data.astype(np.float64)
    y = digits.target.astype(np.int64)
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    conn = sqlite3.connect(DB_PATH)
    conn.execute('CREATE TABLE digits (id INTEGER PRIMARY KEY, features BLOB NOT NULL, label INTEGER NOT NULL)')
    conn.executemany('INSERT INTO digits VALUES (?, ?, ?)',
                     [(i, X[i].tobytes(), int(y[i])) for i in range(len(y))])
    conn.commit(); conn.close()
    np.savetxt(CSV_PATH, np.hstack([X, y.reshape(-1, 1)]), delimiter=',', fmt='%g')
    print(f'Stored {len(y)} samples in {DB_PATH}')
    print(f'Exported {CSV_PATH} for the C version')

build_store()

## 2. Load into RAM and build the index

- `exact` is a hash map: bytes of the feature vector to label. Lookup is O(1).
- `tree` is a sklearn `KDTree`. Nearest-neighbor query is O(log n) average for low-medium d.

In [ ]:
class DigitMemory:
    def __init__(self):
        conn = sqlite3.connect(DB_PATH)
        rows = conn.execute('SELECT features, label FROM digits').fetchall()
        conn.close()
        self.X = np.vstack([np.frombuffer(b, dtype=np.float64) for b, _ in rows])
        self.y = np.array([lab for _, lab in rows], dtype=np.int64)
        self.exact = {self.X[i].tobytes(): int(self.y[i]) for i in range(len(self.y))}
        self.tree = KDTree(self.X)

    def query(self, arr):
        q = np.asarray(arr, dtype=np.float64).reshape(-1)
        if q.size != N_FEATURES:
            raise ValueError(f'expected {N_FEATURES} features, got {q.size}')
        hit = self.exact.get(q.tobytes())
        if hit is not None:
            return hit, 0.0, True
        dist, idx = self.tree.query(q.reshape(1, -1), k=1)
        return int(self.y[idx[0, 0]]), float(dist[0, 0]), False

mem = DigitMemory()
print(f'Loaded {len(mem.y)} samples')

## 3. Demo queries

Exact match hits the hash map. Perturbed and noisy queries fall through to the KDTree.

In [ ]:
def time_query(mem, arr, tag):
    t0 = time.perf_counter()
    label, dist, exact = mem.query(arr)
    dt_us = (time.perf_counter() - t0) * 1e6
    print(f'{tag:<18} label={label}  dist={dist:.4f}  exact={exact}  ({dt_us:.1f} us)')

sample = mem.X[42]
time_query(mem, sample, 'exact')

perturbed = sample.copy(); perturbed[0] += 0.7; perturbed[15] -= 1.3
time_query(mem, perturbed, 'perturbed')

rng = np.random.default_rng(0)
time_query(mem, mem.X[100] + rng.normal(0, 2.0, N_FEATURES), 'noisy')

## 4. Visualize a query and its nearest neighbor

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
query = mem.X[300] + rng.normal(0, 3.0, N_FEATURES)
label, dist, exact = mem.query(query)
_, idx = mem.tree.query(query.reshape(1, -1), k=1)
nn = mem.X[idx[0, 0]]

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
axes[0].imshow(query.reshape(8, 8), cmap='gray_r'); axes[0].set_title('query')
axes[1].imshow(nn.reshape(8, 8),    cmap='gray_r'); axes[1].set_title(f'NN label={label}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'distance = {dist:.4f}, exact = {exact}')